## Import Statements

In [1]:
import os 
from openai import OpenAI
from dotenv import load_dotenv
from pinecone import ServerlessSpec , Pinecone
load_dotenv()
openai_api = os.getenv("open_ai_api")
pinecone_api = os.getenv("pinecone_api")
client = OpenAI(api_key=openai_api)
pc = Pinecone(api_key=pinecone_api)

## Setting up Pinecone

In [2]:
index_name = "genai-v1"
cloud = os.environ.get('PINECONE_CLOUD') or 'aws'
region = os.environ.get('PINECONE_REGION') or 'us-east-1'

spec = ServerlessSpec(cloud=cloud, region=region)

## Fetching Index

In [3]:
import time

# Check the index is available or not  
if index_name not in pc.list_indexes().names():
    # Index is not availbale 
    raise ValueError("There is no Index Available")

index = pc.Index("genai-v1")
# Describe index 
index.describe_index_stats()

DescribeIndexStatsResponse(dimension=1536, total_vector_count=11, metric='cosine', namespaces=2)

## Query

In [ ]:
def display_result(query_result):
    for match in query_result['matches']:
        print(f"ID: {match['id']}, Score: {match['score']}")
        if 'metadata' in match and 'text' in match['metadata']:
            text = match['metadata']['text']
            target_id = query_result['matches'][0]['id']
        else:
            print("No metadata available")
    return text,target_id

In [5]:
embedding_model = "text-embedding-3-small"
def get_embedding(text , model = embedding_model):
    text = text.replace("\n" , " ")
    response = client.embeddings.create(input = [text] , model = model)
    embedding = response.data[0].embedding
    return embedding

In [6]:
def get_query_result(query_text , namespace):
    # Generate The query Vector from the query text
    query_vector = get_embedding(query_text)

    # Perform the query
    query_result = index.query(
        vector = query_vector,
        namespace=namespace,
        top_k=1,
        include_metadata=True
    )

    return query_result

In [9]:
def query_vector_store(query_text , namespace):
    print("Quering the vector Store")

    # REtrive the query results
    query_result = get_query_result(query_text , namespace)

    print("Processed Query Result")
    text , target_id = display_result(query_result)

    return text, target_id

## Retriving Pinecone index

In [10]:
namespace = 'genaisys'
query_text = "The customers like the idea of travelling and learning. Provide your sentiment."

# Call the query Function
text, target_id = query_vector_store(query_text , namespace)

# Display the final output
print("Final output:")
print(f"Text: {text}")
print(f"Target ID: {target_id}")

Quering the vector Store
Processed Query Result
ID: 2, Score: 0.220102862
Final output:
Text: 200,Sentiment analysis  Read the content return a sentiment analysis nalysis on this text and provide a score with the label named : Sentiment analysis score followed by a numerical value between 0 and 1  with no + or - sign and  add an explanation to justify the score.
Target ID: 2


## Retriving The Data

In [11]:
namespace = 'data01'
query_text = "What did the CTO say about the different types of memory?"

text,target_id = query_vector_store(query_text,namespace=namespace)

# Display the final output
print("Final output:")
print(f"Text: {text}")
print(f"Target ID: {target_id}")
     

Quering the vector Store
Processed Query Result
ID: 3, Score: 0.562109649
Final output:
Text: We defined memoryless, short-term, long-term memory, and cross-topic memory. For the hybrid travel marketing campaign, we will distinguish semantic memory (facts) from episodic memory (personal events in time, for example). The CTO said that we will need to use episodic memories of past customer trips to make the semantic aspects of our trips more engaging. We could, for example, merge geological semantic information with customer experiences to make our advertisements.
Target ID: 3
